## FlyRank Internship | Backend Track W3-A2
### Connecting your CRUD to the database

### Setup

In [ ]:
import sqlite3
import os

from fastapi import FastAPI, Response
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

DB_PATH = "tasks.db"

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

print("Imports OK. Working with:", os.path.abspath(DB_PATH))


Imports OK. Working with: c:\Users\Muhammad Razi\workspace_python\AIAutomation\FlyRankAI\WEEK_3\tasks.db


### Task 0 - Set up SQLite
*"Your list gets a permanent home."*

Connect to a file that doesn't exist yet. SQLite creates it automatically, then create
the `tasks` table **if it doesn't already exist**, so re-running this on a real server never
wipes existing data.

In [9]:
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.row_factory = sqlite3.Row  # lets us read columns by name, e.g. row["title"]

conn.execute('''
    CREATE TABLE IF NOT EXISTS tasks (
        id    INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT    NOT NULL,
        done  INTEGER NOT NULL DEFAULT 0
    )
''')
conn.commit()

# checkpoint
assert os.path.exists(DB_PATH), "tasks.db should exist on disk now"
tables = [r["name"] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")]
print("tasks.db exists:", os.path.exists(DB_PATH))
print("Tables:", tables)
assert "tasks" in tables

print("\nTask 0 checkpoint completed.")


tasks.db exists: True
Tables: ['tasks', 'sqlite_sequence']

Task 0 checkpoint completed.


## Task 1 - Seed & read

In [12]:
def seed_if_empty() -> bool:
    
    # Insert the 3 example tasks only if the table is currently empty.
    # Returns whether it seeded.

    count = conn.execute("SELECT COUNT(*) AS c FROM tasks").fetchone()["c"]
    if count > 0:
        return False

    with conn:  # transaction: all 3 inserts succeed together, or none do
        conn.executemany(
            "INSERT INTO tasks (title, done) VALUES (?, ?)",
            [("Buy milk", 0), ("Write weekly report", 1), ("Walk the dog", 0)],
        )
    return True


# Clear any old data before running the checkpoint so the test is deterministic
# when this notebook is re-executed.
conn.execute("DELETE FROM tasks")
conn.execute("DELETE FROM sqlite_sequence WHERE name = 'tasks'")
conn.commit()

first_call = seed_if_empty()
second_call = seed_if_empty()  # table is no longer empty -> should be a no-op

rows = conn.execute("SELECT * FROM tasks ORDER BY id").fetchall()
print("First call seeded  :", first_call)
print("Second call seeded :", second_call)
print("Row count           :", len(rows))
for r in rows:
    print(" ", dict(r))

assert first_call is True
assert second_call is False
assert len(rows) == 3

print("\nTask 1 checkpoint completed - seeding is idempotent.")


First call seeded  : True
Second call seeded : False
Row count           : 3
  {'id': 1, 'title': 'Buy milk', 'done': 0}
  {'id': 2, 'title': 'Write weekly report', 'done': 1}
  {'id': 3, 'title': 'Walk the dog', 'done': 0}

Task 1 checkpoint completed - seeding is idempotent.


## Task 2 - Read endpoints against SQL
*"Same routes, new storage."*

`GET /tasks` and `GET /tasks/{id}` now run `SELECT` queries instead of scanning a list.
Every value from the client (the `task_id` in the URL, `search`/`done` in the query string)
goes in as a **parameter** — a `?` placeholder — never pasted into the SQL text. That's what
keeps user input from being able to corrupt or hijack a query.

In [15]:
def error(status_code: int, message: str) -> JSONResponse:
    return JSONResponse(status_code=status_code, content={"error": message})

def row_to_task(row: sqlite3.Row) -> dict:
    return {"id": row["id"], "title": row["title"], "done": bool(row["done"])}

app = FastAPI(title="Task API", version="2.0")

@app.get("/")
def root():
    return {"name": "Task API", "version": "2.0", "storage": "SQLite", "endpoints": ["/tasks"]}

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/tasks")
def list_tasks(done: Optional[bool] = None, search: Optional[str] = None,
               limit: Optional[int] = None, offset: int = 0):
    query = "SELECT * FROM tasks WHERE 1=1"
    params: list = []
    if done is not None:                      # GET /tasks?done=true
        query += " AND done = ?"
        params.append(1 if done else 0)
    if search:                                 # GET /tasks?search=milk
        query += " AND title LIKE ?"
        params.append(f"%{search}%")
    query += " ORDER BY id"
    if limit is not None:                      # GET /tasks?limit=2&offset=1
        query += " LIMIT ? OFFSET ?"
        params.extend([limit, offset])
    rows = conn.execute(query, params).fetchall()
    return [row_to_task(r) for r in rows]

@app.get("/tasks/{task_id}")
def get_task(task_id: int):
    row = conn.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()
    if row is None:
        return error(404, f"Task {task_id} not found")
    return row_to_task(row)

client = TestClient(app)

r1 = client.get("/tasks")
print("GET /tasks          -", r1.status_code, r1.json())
assert r1.status_code == 200 and len(r1.json()) == 3

r2 = client.get("/tasks/1")
print("GET /tasks/1        -", r2.status_code, r2.json())
assert r2.status_code == 200

r3 = client.get("/tasks/99")
print("GET /tasks/99       -", r3.status_code, r3.json())
assert r3.status_code == 404

r4 = client.get("/tasks", params={"done": "true"})
print("GET /tasks?done=true-", r4.status_code, r4.json())
assert all(t["done"] for t in r4.json())

print("\nTask 2 checkpoint completed - reads hit real SQL, parameterized.")

GET /tasks          - 200 [{'id': 1, 'title': 'Buy milk', 'done': False}, {'id': 2, 'title': 'Write weekly report', 'done': True}, {'id': 3, 'title': 'Walk the dog', 'done': False}]
GET /tasks/1        - 200 {'id': 1, 'title': 'Buy milk', 'done': False}
GET /tasks/99       - 404 {'error': 'Task 99 not found'}
GET /tasks?done=true- 200 [{'id': 2, 'title': 'Write weekly report', 'done': True}]

Task 2 checkpoint completed - reads hit real SQL, parameterized.


In [ ]:
# commiting checkpoint to git